# Two-Step Task Model Fitting

**Workflow:** Change the config cell below, then run all cells.

| Model name | Family | Parameters |
|---|---|---|
| `mf` | Model-Free | alpha, forget, lambda, Wmf |
| `mf_p` | Model-Free | + P (single perseveration) |
| `mf_pmulti` | Model-Free | + P, alpha_c (EMA perseveration) |
| `p_only` | Model-Free | P only (no RL) |
| `mb` | Model-Based | alpha, forget, lambda, Wmb |
| `mb_p` | Model-Based | + P |
| `mb_pmulti` | Model-Based | + P, alpha_c |
| `mb_p_asym` | Model-Based | alpha_r, alpha_n (asymmetric) + P |
| `mb_p_vinherit` | Model-Based | + P, value inheritance across sessions |
| `mb_p_vasym` | Model-Based | value inheritance + asymmetric + P |
| `hyb` | Hybrid | alpha, forget, lambda, Wmf, Wmb |
| `hyb_p` | Hybrid | + P |
| `hyb_pmulti` | Hybrid | + P, alpha_c |

In [6]:
# ============================================================
# CONFIGURATION — change MODEL here, then run all cells
# ============================================================
# Available models: mf, mf_p, mf_pmulti, p_only,
#                   mb, mb_p, mb_pmulti, mb_p_asym, mb_p_vinherit, mb_p_vasym,
#                   hyb, hyb_p, hyb_pmulti,
#                   latent_state, reward_as_cue
MODEL = "mb_p_asym"  # change this to switch models

FORCE_REFIT = True  # set True to ignore cached CSVs and refit everything

In [7]:
import sys, os
# Resolve organized/ robustly regardless of where Jupyter was launched
_d = os.path.abspath(os.getcwd())
for _ in range(5):
    if os.path.isfile(os.path.join(_d, "config.py")):
        break
    _d = os.path.dirname(_d)
sys.path.insert(0, os.path.join(_d, "src"))
sys.path.insert(0, _d)

import numpy as np
import pandas as pd
import pystan
import nest_asyncio
nest_asyncio.apply()

import data_import as di
from models import get_model, list_models
from fitting import RL_data_arrange, LOO
from config import SUBJECTS, cv_result_csv, raw_subject_dir

list_models()

Name                      Family       Description
----------------------------------------------------------------------
  hyb                     hybrid       Hybrid (MF+MB) baseline
  hyb_inf                 hybrid       Asymmetric Inference + Model-Free + single perseveration
  hyb_p                   hybrid       Hybrid + single perseveration
  hyb_pmulti              hybrid       Hybrid + multi-trial perseveration (EMA)
  ls                      Latent State Latent State (Akam 2015, reduced task) — Bayes + posterior-weighted ε-greedy
  ls_asym                 Latent State Latent State Assymetry (no perseveration)
  ls_asym_p               Latent State Latent State Assymetry + P (single perseveration)
  ls_asym_pmulti          Latent State Latent State Assymetry + P multi (EMA perseveration)
  ls_p                    Latent State Latent State (Akam 2015) + perseveration
  ls_pmulti               Latent State alias for ls_asym_pmulti
  mb                      model_based  Model-bas

In [8]:
# Compile Stan model once (shared across all stages)
model = get_model(MODEL)
print(f'Model  : {model.name} — {model.description}')
print(f'Params : {model.param_names}')

sm = pystan.StanModel(model_code=model.stan_code)
print('Stan compilation complete.')

Model  : mb_p_asym — Model-based + perseveration, asymmetric learning rates
Params : ['alpha_r', 'alpha_n', 'forget', 'Wmb', 'P']


In file included from /gpfs/radev/home/my483/.conda/envs/stanenv/lib/python3.8/site-packages/numpy/core/include/numpy/ndarraytypes.h:1940,
                 from /gpfs/radev/home/my483/.conda/envs/stanenv/lib/python3.8/site-packages/numpy/core/include/numpy/ndarrayobject.h:12,
                 from /gpfs/radev/home/my483/.conda/envs/stanenv/lib/python3.8/site-packages/numpy/core/include/numpy/arrayobject.h:5,
                 from /tmp/pystan_795z0yrp/stanfit4anon_model_d2104bf24a46780bb28c78abde6ed9d7_7218243030389613493.cpp:1315:
/gpfs/radev/home/my483/.conda/envs/stanenv/lib/python3.8/site-packages/numpy/core/include/numpy/npy_1_7_deprecated_api.h:17:2: warning: #warning "Using deprecated NumPy API, disable it with " "#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION" [-Wcpp]
   17 | #warning "Using deprecated NumPy API, disable it with " \
      |  ^~~~~~~
In file included from /tmp/pystan_795z0yrp/stanfit4anon_model_d2104bf24a46780bb28c78abde6ed9d7_7218243030389613493.cpp:1322:
/gp

Stan compilation complete.


In [9]:
def load_sessions(subject, training_stage):
    if training_stage in ("4.6", "4.7"):
        exp = di.Experiment(raw_subject_dir(subject, training=True))
        exp.save()
        return [s for s in exp.get_sessions(subject_IDs="all") if s.training_stage == training_stage]
    else:
        exp = di.Experiment(raw_subject_dir(subject, training=False))
        exp.save()
        return exp.get_sessions(subject_IDs="all")

# Run LOO for all three stages
STAGES = [("4.6", "4.6"), ("4.7", "4.7"), (None, "4.8")]  # (training_stage, label)

all_stage_dfs = {}

for training_stage, stage_lbl in STAGES:
    output_csv = cv_result_csv(MODEL, stage_lbl)

    if not FORCE_REFIT and os.path.exists(output_csv):
        print(f'\n=== Stage {stage_lbl}: loading from {output_csv} ===')
        all_stage_dfs[stage_lbl] = pd.read_csv(output_csv)
        continue

    print(f'\n=== Stage {stage_lbl}: fitting {MODEL} ===')

    # Load data for this stage
    mice_data = {}
    for subject in SUBJECTS:
        sessions = load_sessions(subject, training_stage)
        if not sessions:
            print(f"  {subject}: no sessions found, skipping")
            continue
        S, T, c, ss, tt, r, PR, PL = RL_data_arrange(sessions)
        mice_data[subject] = (S, T, max(T), c, ss, tt, r, PR, PL)
        print(f"  {subject}: {S} sessions")

    # Run LOO
    rows = []
    for subject, data in mice_data.items():
        S, T, T_max, c, ss, tt, r, PR, PL = data
        print(f'  --- {subject} (S={S}) ---')
        nl, param = LOO(model, sm, S, T, T_max, c, ss, tt, r, PL, PR)
        row = {'Subject': subject, 'Cross validation': np.exp(np.mean(nl))}
        for p_name in model.param_names:
            if p_name in param:
                row[p_name] = param[p_name]
        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    all_stage_dfs[stage_lbl] = df
    print(f'  Mean CV: {df["Cross validation"].mean():.4f}  →  saved to {output_csv}')

print('\nDone.')


=== Stage 4.6: fitting mb_p_asym ===
Saved sessions loaded from: sessions.pkl
  WT1: 5 sessions
Saved sessions loaded from: sessions.pkl
  WT2: 11 sessions
Saved sessions loaded from: sessions.pkl
  WT3: 11 sessions
Saved sessions loaded from: sessions.pkl
  WT4: 6 sessions
Saved sessions loaded from: sessions.pkl
  WT5: 9 sessions
Saved sessions loaded from: sessions.pkl
  WT6: 2 sessions
  --- WT1 (S=5) ---
Initial log joint probability = -468.9
    Iter      log prob        ||dx||      ||grad||       alpha      alpha0  # evals  Notes 
      19      -418.101     0.0270183      0.065409          10           1       22   
    Iter      log prob        ||dx||      ||grad||       alpha      alpha0  # evals  Notes 
      23      -418.101    0.00472072    0.00228434      0.8869      0.8869       26   
Optimization terminated normally: 
  Convergence detected: relative gradient magnitude is below tolerance
Initial log joint probability = -454.108
    Iter      log prob        ||dx||      

In [10]:
# Summary table: subjects × stages
summary = pd.concat(
    [df.set_index('Subject')[['Cross validation']].rename(columns={'Cross validation': lbl})
     for lbl, df in all_stage_dfs.items()],
    axis=1
)
summary.loc['Mean'] = summary.mean()
summary

,4.6,4.7,4.8
Subject,,,
WT1,0.496808,0.547333,0.593228
WT2,0.499193,0.506010,0.578103
WT3,0.507362,0.526329,0.566957
WT4,0.494236,0.521654,0.595072
WT5,0.502495,0.536299,0.567363
WT6,0.539490,0.563936,0.632381
Mean,0.506597,0.533594,0.588851
